# 👑🎨 KING2-IMAGE — تدريب مولّد صور (SDXL + LoRA) — Kaggle

تدريب LoRA فوق Stable Diffusion XL على عيّنة من `jackyhate/text-to-image-2M`،
ثم رفع الأوزان + Model Card إلى `RASHID778/king2-image`.

المتطلبات: GPU + Internet + سر `HF_TOKEN` (صلاحية Write).

In [ ]:
# 1. فحص وجود التوكن أولاً — فشل سريع، وبدون استيراد مكتبات HF
# (الاستيراد قبل الترقية يترك موديول قديم في الذاكرة ويكسر datasets لاحقاً)
import os
from pathlib import Path

HF_TOKEN = None
_secret_file = Path('/kaggle/input/king2-secrets/hf_token.txt')
if _secret_file.exists():
    HF_TOKEN = _secret_file.read_text().strip()
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN')
assert HF_TOKEN, '❌ HF_TOKEN غير متاح — شغّل tools/upload_hf_token_to_kaggle.py أو اربط السر من Add-ons → Secrets'
os.environ['HF_TOKEN'] = HF_TOKEN
print('✅ التوكن موجود — المصادقة الفعلية بعد التثبيت')

In [ ]:
# 2. تثبيت المكتبات + سكربت التدريب الرسمي
# diffusers من المصدر لأن سكربت التدريب (main) يحتاج نسخة dev
!pip install -q git+https://github.com/huggingface/diffusers
!pip install -qU accelerate transformers peft bitsandbytes datasets huggingface_hub webdataset
# wandb معطوب ويوقف السكربت — وtorchao القديم في صورة Kaggle يكسر استيراد diffusers dev
!pip uninstall -y wandb torchao -q
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/examples/text_to_image/train_text_to_image_lora_sdxl.py -O train_text_to_image_lora_sdxl.py
!sed -i 's/is_wandb_available()/False/g' train_text_to_image_lora_sdxl.py
!sed -i 's/log_with=args.report_to/log_with=None/g' train_text_to_image_lora_sdxl.py
print('✅ تم التثبيت')

In [ ]:
# 2.5 تسجيل الدخول HuggingFace — استيراد نظيف بعد الترقية
import os
from huggingface_hub import login, whoami
login(os.environ['HF_TOKEN'])
print('Logged in as:', whoami()['name'])

In [ ]:
# 3. بث عيّنة من الداتا وحفظها كـ imagefolder
import os, json
from datasets import load_dataset

N_SAMPLES = 3000
RES = 1024
DATA_DIR = '/kaggle/working/king2_img_data'
os.makedirs(DATA_DIR, exist_ok=True)

ds = load_dataset('jackyhate/text-to-image-2M', split='train', streaming=True)
meta, kept = [], 0
for ex in ds:
    if kept >= N_SAMPLES:
        break
    try:
        img = ex['jpg'].convert('RGB')
        prompt = (ex.get('json') or {}).get('prompt', '').strip()
        if not prompt:
            continue
        w, h = img.size; s = min(w, h)
        img = img.crop(((w-s)//2,(h-s)//2,(w-s)//2+s,(h-s)//2+s)).resize((RES, RES))
        fn = f'{kept:06d}.jpg'
        img.save(os.path.join(DATA_DIR, fn), quality=92)
        meta.append({'file_name': fn, 'prompt': prompt}); kept += 1
        if kept % 250 == 0:
            print(f'  حفظ {kept}/{N_SAMPLES}')
    except Exception:
        continue

with open(os.path.join(DATA_DIR, 'metadata.jsonl'), 'w', encoding='utf-8') as f:
    for m in meta:
        f.write(json.dumps(m, ensure_ascii=False) + '\n')
print(f'✅ جاهز: {kept} صورة')

In [ ]:
# 5. إعداد accelerate — GPU واحد فقط (T4)
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from accelerate.utils import write_basic_config
write_basic_config(mixed_precision='fp16')
print('✅ accelerate config (single GPU)')

In [ ]:
# 5. التدريب — SDXL LoRA (الإعدادات المُثبتة على 16GB)
!accelerate launch train_text_to_image_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --pretrained_vae_model_name_or_path="madebyollin/sdxl-vae-fp16-fix" \
  --train_data_dir="/kaggle/working/king2_img_data" \
  --caption_column="prompt" \
  --resolution=768 --center_crop --random_flip \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --gradient_checkpointing \
  --use_8bit_adam \
  --mixed_precision="fp16" \
  --max_train_steps=1500 \
  --learning_rate=1e-4 \
  --lr_scheduler="cosine" --lr_warmup_steps=0 \
  --rank=16 \
  --checkpointing_steps=500 \
  --seed=42 \
  --output_dir="/kaggle/working/king2-image-lora" \
  --report_to="tensorboard" 2>&1 | tee /kaggle/working/train.log

import os
_w = '/kaggle/working/king2-image-lora/pytorch_lora_weights.safetensors'
if os.path.exists(_w):
    print('\n✅✅ التدريب نجح — ملف الأوزان موجود:', _w)
else:
    print('\n❌❌ التدريب فشل — آخر 40 سطر:')
    print('='*60)
    !tail -n 40 /kaggle/working/train.log
    raise SystemExit('training failed')

In [ ]:
# 6. اختبار النموذج — توليد صورة
import torch
from diffusers import DiffusionPipeline

pipe = DiffusionPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0', torch_dtype=torch.float16
).to('cuda')
pipe.load_lora_weights('/kaggle/working/king2-image-lora', weight_name='pytorch_lora_weights.safetensors')
image = pipe('a futuristic royal palace at sunset, highly detailed, 8k, cinematic',
             num_inference_steps=30, guidance_scale=7.0).images[0]
image.save('/kaggle/working/king2_test.png')
del pipe
torch.cuda.empty_cache()
print('✅ تم توليد صورة اختبارية')
image

In [ ]:
# 7. Model Card + رفع إلى HuggingFace باسم king2-image
import os
from huggingface_hub import HfApi, create_repo

REPO_ID = 'RASHID778/king2-image'
LORA_DIR = '/kaggle/working/king2-image-lora'

MODEL_CARD = '''---
license: openrail++
base_model: stabilityai/stable-diffusion-xl-base-1.0
library_name: diffusers
tags:
- text-to-image
- stable-diffusion-xl
- lora
- diffusers
- king2
pipeline_tag: text-to-image
---

# 👑 KING2-IMAGE

SDXL LoRA (rank 16) مدرّب على عيّنة من `jackyhate/text-to-image-2M` — موديل الصور الخاص بمنصة KING2.

## Usage

```python
import torch
from diffusers import DiffusionPipeline

pipe = DiffusionPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16
).to("cuda")
pipe.load_lora_weights("RASHID778/king2-image")
image = pipe("a futuristic royal palace at sunset, highly detailed, 8k").images[0]
```

## Training

- Base: `stabilityai/stable-diffusion-xl-base-1.0` + `madebyollin/sdxl-vae-fp16-fix`
- Data: 3000 صورة من `jackyhate/text-to-image-2M`
- 1500 خطوة، resolution 768، rank 16، lr 1e-4 cosine، fp16
'''

with open(os.path.join(LORA_DIR, 'README.md'), 'w', encoding='utf-8') as f:
    f.write(MODEL_CARD)

create_repo(REPO_ID, repo_type='model', exist_ok=True)
HfApi().upload_folder(
    folder_path=LORA_DIR,
    repo_id=REPO_ID, repo_type='model',
    commit_message='KING2-IMAGE: SDXL LoRA trained on text-to-image-2M subset (Kaggle)',
)
print(f'✅ تم الرفع: https://huggingface.co/{REPO_ID}')